In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import * 

In [0]:
df= spark.read.table('fmcg2.bronze.customers')

In [0]:
display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
df_dupliicates=df.groupBy("customer_id").count().filter(col("customer_id")>1)
display(df_dupliicates)

In [0]:
df_duplicates= df.groupBy('customer_id').count().where('count >1')

In [0]:
display(df_duplicates)

In [0]:
df= df.dropDuplicates(['customer_id'])

In [0]:
display(df.filter(col('customer_name') != trim(col('customer_name'))))

In [0]:
df=df.withColumn('customer_name',trim(col('customer_name')))

In [0]:
display(df.select('city').distinct())

In [0]:
city_mapping={
    "Bengalore":"Bengaluru",
    "Bengaluruu":"Bengaluru",
    "NewDelhee":"New Delhi",
    "NewDelhi":"New Delhi",
    "NewDheli":"New Delh",
    "Hyderabadd":"Hyderabad"

}

In [0]:
allowed=['Bengaluru','New Delhi','Hyderabad']
df=df.replace(city_mapping,subset=['city']).withColumn("city",when(col('city').isNull(),None).when(col("city").isin(allowed),col("city")).otherwise(None))

In [0]:
display(df.select('city').distinct())

In [0]:
display(df.select("customer_name").distinct())

In [0]:
df=df.withColumn('customer_name',initcap(col("customer_name")))

In [0]:
display(df)

In [0]:
mode_city = (
    df.groupBy("city")
      .count()
      .orderBy(col("count").desc())
      .first()
)

print(mode_city)

In [0]:
df=df.fillna({'city':mode_city['city']})

In [0]:
display(df.filter(col('city').isNull()).count())

In [0]:
df.printSchema()

In [0]:
df=df.withColumn('customer_id',col('customer_id').cast('string'))

In [0]:
display(df.withColumn('customer',concat_ws('_','customer_name','city')))

In [0]:

df=df.withColumn(
        'customer',
        concat_ws('_', 'customer_name', 'city')
)

In [0]:
df.drop('city')

In [0]:
display(df)

In [0]:
df=df.drop('city','customer_name')

In [0]:
df=df.withColumn('market',lit('India'))\
    .withColumn('platform',lit('Sports Bar'))\
        .withColumn('Channel',lit('Aquisition'))

In [0]:
df=df.withColumnRenamed('customer_id','customer_code')

In [0]:
df.display()

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable("fmcg2.silver.customer_enr")

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable("fmcg2.Gold.customer_Gold")

Merge the child company table to the parent company 

In [0]:
parent_table=spark.read.table('fmcg.gold.dim_customers')
 

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, 'fmcg.gold.dim_customers')
child_table=spark.read.table('fmcg2.gold.customer_gold') 

target_table.alias('target').merge(
    child_table.alias('source'),
    'target.customer_code= source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()